In [26]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [27]:
import numpy as np
import pandas as pd
from google.colab import drive
from imblearn.over_sampling import SMOTE
from sklearn.metrics import (
    auc,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_curve,
)
from sklearn.naive_bayes import GaussianNB
# (a)
file_path = "/content/drive/MyDrive/data/wpbc.data"


columns = (
    ["ID", "Outcome", "Time"]
    + [f"Feature_{i}" for i in range(1, 31)]
    + ["Tumor_Size", "Lymph_Node_Status"]
)
df = pd.read_csv(file_path, header=None, names=columns)

df["Outcome"] = df["Outcome"].map({"N": 0, "R": 1})

features_to_keep = [col for col in df.columns if col not in ["ID", "Time", "Outcome"]]

# (b)
special_idx = 196
special_row = df.loc[special_idx]
special_outcome = special_row["Outcome"]

df_remaining = df.drop(index=special_idx)
n_indices_pool = df_remaining[df_remaining["Outcome"] == 0].index.tolist()
r_indices_pool = df_remaining[df_remaining["Outcome"] == 1].index.tolist()

if special_outcome == 0:
    train_indices = n_indices_pool[:129] + r_indices_pool[:37] + [special_idx]
else:
    train_indices = n_indices_pool[:130] + r_indices_pool[:36] + [special_idx]

test_indices = [i for i in df.index if i not in train_indices]

X_train = df.loc[train_indices, features_to_keep].copy()
y_train = df.loc[train_indices, "Outcome"].copy()
X_test = df.loc[test_indices, features_to_keep].copy()
y_test = df.loc[test_indices, "Outcome"].copy()

print(f"train:\n{y_train.value_counts()}")
print(f"test:\n{y_test.value_counts()}")

# (c)
df_train_raw = df.loc[train_indices]
missing_rows = df_train_raw.loc[
    df_train_raw["Lymph_Node_Status"] == "?",
    ["ID", "Outcome", "Tumor_Size", "Lymph_Node_Status"]
]
print("Missing Instances in Training Set")
print(missing_rows)
X_train["Lymph_Node_Status"] = pd.to_numeric(X_train["Lymph_Node_Status"], errors="coerce")
X_test["Lymph_Node_Status"] = pd.to_numeric(X_test["Lymph_Node_Status"], errors="coerce")

median_lymph = X_train["Lymph_Node_Status"].median()
print(f"\nCalculated Training Set Median: {median_lymph}")

X_train["Lymph_Node_Status"] = X_train["Lymph_Node_Status"].fillna(median_lymph)
X_test["Lymph_Node_Status"] = X_test["Lymph_Node_Status"].fillna(median_lymph)

print("\n[Success] Missing values handled. You can now safely run the (d)i and (d)ii models!")
# (d) i.  basic bayes
gnb_base = GaussianNB()
gnb_base.fit(X_train, y_train)
evaluate_model(gnb_base, X_train, y_train, X_test, y_test, title="Base Naïve Bayes")

# (d) ii. Downsampling + SMOTE
train_df = X_train.copy()
train_df["Outcome"] = y_train

df_n = train_df[train_df["Outcome"] == 0]
df_r = train_df[train_df["Outcome"] == 1]

df_n_downsampled = df_n.sample(n=90, random_state=42)
df_combined = pd.concat([df_n_downsampled, df_r])

X_combined = df_combined.drop(columns=["Outcome"])
y_combined = df_combined["Outcome"]

smote = SMOTE(sampling_strategy={0: 90, 1: 90}, k_neighbors=5, random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_combined, y_combined)

gnb_balanced = GaussianNB()
gnb_balanced.fit(X_train_balanced, y_train_balanced)
evaluate_model(
    gnb_balanced,
    X_train_balanced,
    y_train_balanced,
    X_test,
    y_test,
    title="Balanced Naïve Bayes (SMOTE + Downsampling)",
)

train:
Outcome
0    130
1     37
Name: count, dtype: int64
test:
Outcome
0    21
1    10
Name: count, dtype: int64
Missing Instances in Training Set
         ID  Outcome  Tumor_Size Lymph_Node_Status
6    844359        0         1.5                 ?
28   854253        0         1.5                 ?
85   877500        0         1.5                 ?
196  947204        1         3.0                 ?

Calculated Training Set Median: 1.0

[Success] Missing values handled. You can now safely run the (d)i and (d)ii models!

=============== Base Naïve Bayes results ===============
[Train Set]
Confusion Matrix:
[[99 31]
 [20 17]]
Precision: 0.3542 | Recall: 0.4595 | F1-Score: 0.4000 | AUC: 0.6736

[Test Set]
Confusion Matrix:
[[14  7]
 [ 7  3]]
Precision: 0.3000 | Recall: 0.3000 | F1-Score: 0.3000 | AUC: 0.5429


=============== Balanced Naïve Bayes (SMOTE + Downsampling) results ===============
[Train Set]
Confusion Matrix:
[[61 29]
 [33 57]]
Precision: 0.6628 | Recall: 0.6333 | F1-Score: 